# Maktspråk — retrain the party classifier on the rebuilt corpus (Colab GPU)

Trains on `speeches_full.parquet` (the rebuilt source-of-truth corpus), filtered to
**2015-2026**, the model's deliberate scope, not the trimmed DB. Pre-2015 exists in the
corpus for the drift/time-series features only and is kept out of training (era drift,
no SD before 2010). This matches the model to the corpus
the site serves and gives a clean, reconstructable speaker split
(`val_speakers.json`). Run on a **GPU runtime** (Runtime → Change runtime type → GPU).

In [ ]:
# 1. GPU + mount Drive (checkpoints survive disconnects there)
!nvidia-smi -L
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
# 2. HF token -> environment (pull the corpus, push the model). Colab Secrets or paste.
import os

try:
    from google.colab import userdata

    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception:
    import getpass

    os.environ["HF_TOKEN"] = getpass.getpass("HF_TOKEN: ")
print("HF_TOKEN set:", bool(os.environ.get("HF_TOKEN")))

In [ ]:
# 3. Clone the repo and install dependencies
%cd /content
!rm -rf maktsprak
!git clone https://github.com/MartinBlomqvistDev/maktsprak.git
%cd maktsprak
!pip install -q -r requirements.txt

In [ ]:
# 3b. Pull the rebuilt corpus (75,148 speeches) from the private HF dataset
import os
from huggingface_hub import hf_hub_download

os.makedirs("data/parquet", exist_ok=True)
hf_hub_download(
    "MartinBlomqvist/maktsprak-corpus",
    "speeches_full.parquet",
    repo_type="dataset",
    local_dir="data/parquet",
    token=os.environ["HF_TOKEN"],
)
import pandas as pd

print("corpus rows:", len(pd.read_parquet("data/parquet/speeches_full.parquet", columns=["party"])))

In [ ]:
# (optional) wipe old checkpoints for a clean run
!rm -rf /content/drive/MyDrive/MaktsprakAI_Checkpoints

In [ ]:
# 4. Train on the rebuilt corpus, 2015-2026 (the model's deliberate scope; persists
#    val_speakers.json for THIS corpus). Pre-2015 is in the corpus for the drift/time-series
#    features only: era drift, no SD before 2010, FP->L rename, so it stays OUT of training.
#    Cheap baseline: add --no-fgm --max-length 256.  Disconnect-safe: re-run, it resumes from Drive.
%cd /content/maktsprak
!python scripts/train_party_model_db.py     --parquet data/parquet/speeches_full.parquet     --year-min 2015     --output-dir /content/drive/MyDrive/MaktsprakAI_Checkpoints

In [ ]:
# 5. Honest speaker-independent number on the NEW val set (same corpus + persisted split).
#    (The old live model is era-confounded on the full corpus, so it is not a fair row here.)
!python scripts/evaluate_model.py     --model /content/drive/MyDrive/MaktsprakAI_Checkpoints     --val-speakers /content/drive/MyDrive/MaktsprakAI_Checkpoints/val_speakers.json     --parquet data/parquet/speeches_full.parquet     --year-min 2015     --limit 0
import glob
from IPython.display import Image, display

for path in sorted(glob.glob("results/confusion_*.png")):
    display(Image(path))

In [ ]:
# 6. Push the retrained model to a NEW HF repo for A/B (does NOT touch the live model).
#    Promote to maktsprak_classifier_clean only after it is verified locally.
import os
from huggingface_hub import HfApi

api = HfApi(token=os.environ["HF_TOKEN"])
api.upload_folder(
    folder_path="/content/drive/MyDrive/MaktsprakAI_Checkpoints",
    repo_id="MartinBlomqvist/maktsprak_classifier_reindexed",
    repo_type="model",
    commit_message="Retrain on rebuilt corpus (75,148, 2002-2026), speaker-independent",
    ignore_patterns=["checkpoints/*", "*.pt"],
)
print("Pushed to MartinBlomqvist/maktsprak_classifier_reindexed (val_speakers.json included)")